# NpuKit — MNIST tiny-ViT (PYNQ-Z2)

Geometry: native **28×28**, patch **7** → **T=16**, patch vec **49→pad56**, **D=8**.

Requires glue bitstream with Softmax `len==MAX_LEN` fix.

Train on the Docker host (torch):
```bash
python3 host/train_vit_mnist.py
```
Then copy `vit_mnist_weights.npz`, `mnist_sample.npz`, `npukit.bit`, and this notebook to the board.

This notebook checks **ref vs board** match and batch accuracy (not 100% classification).

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_vit_mnist as vit

importlib.reload(vit)
print("T", vit.VIT_T, "D", vit.VIT_D, "patch_dim", vit.PATCH_DIM_RAW, "->", vit.PATCH_DIM)
print("weights", vit.DEFAULT_WEIGHTS, "exists", vit.DEFAULT_WEIGHTS.exists())
print("sample", vit.DEFAULT_SAMPLE, "exists", vit.DEFAULT_SAMPLE.exists())

T 16 D 8 patch_dim 49 -> 56
weights /home/xilinx/jupyter_notebooks/vit_mnist_weights.npz exists True
sample /home/xilinx/jupyter_notebooks/mnist_sample.npz exists True


## Offline ref (trained weights + real MNIST sample)

In [2]:
rc = vit.run_vit_smoke(bit_path=None, seed=0, n=64)
assert rc == 0
print("ref-only return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=64)
=== MNIST tiny-ViT smoke ===
IMG=28 PATCH=7 T=16 D=8 patch_dim=49->pad56 classes=10
scales ACT/W/P=12.31/85.04/180.71
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz

--- image[0] label=4 ---
--- ref ---
ref pred=4 logits_q12[:4]=[-31662, -47374, -5224, 25624]

--- image[1] label=3 ---
--- ref ---
ref pred=2 logits_q12[:4]=[-13442, -21108, 26356, 23808]

--- image[2] label=1 ---
--- ref ---
ref pred=6 logits_q12[:4]=[5224, 19101, 6167, -13552]

--- image[3] label=2 ---
--- ref ---
ref pred=2 logits_q12[:4]=[-27479, -540, 26219, 18561]

--- image[4] label=3 ---
--- ref ---
ref pred=3 logits_q12[:4]=[-31259, -26669, 13810, 37978]

--- image[5] label=5 ---
--- ref ---
ref pred=5 logits_q12[:4]=[24364, -19527, -6394, 13900]

--- image[6] label=0 ---
--- ref ---
ref pred=0 logits_q12[:4]=[27804, -20611, -1628, 8500]

--- image[7] label=9 ---
--- ref ---
ref pred=3 logits_q12[:4]=[-26790, -704, 11611

## Board: ref vs FPGA + batch accuracy

In [3]:
rc = vit.run_vit_smoke(bit_path=BIT, seed=0, n=64)
assert rc == 0
print("board return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=64)
=== MNIST tiny-ViT smoke ===
IMG=28 PATCH=7 T=16 D=8 patch_dim=49->pad56 classes=10
scales ACT/W/P=12.31/85.04/180.71
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz


Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID=0x4E50554B version=0x00000300 features=0x00000003

--- image[0] label=4 ---
--- ref ---
ref pred=4 logits_q12[:4]=[-31662, -47374, -5224, 25624]
--- FPGA ---
hw  pred=4 logits_q12[:4]=[-31662, -47374, -5224, 25624]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=446  tol=1024
logits: PASS  max|err|=0  tol=1024

--- image[1] label=3 ---
--- ref ---
ref pred=2 logits_q12[:4]=[-13442, -21108, 26356, 23808]
--- FPGA ---
hw  pred=2 logits_q12[:4]=[-13775, -21335, 26481, 24012]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=309  tol=1024
logits: PASS  max|err|=333  tol=1024

--- image[2] label=1 ---
--- ref ---
ref pred=6 logits_q12[:4]=[5224, 19101, 6167, -13552]
--- FPGA ---
hw  pred=6 logits_q12[:4]=[4884, 18893, 6308, -13141]
tokens: PASS  max|err|=0  tol=512
block.y_out: PASS  max|err|=297  tol=1024
logits: PASS  max|err|=411  tol=1024

--- image[3] label=2 ---
--- ref ---
ref pred=2 logi